In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import json

from dotenv import load_dotenv
load_dotenv()

import ollama

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import InjectedState

from file_tools import (
    DeepAgentState,
    ls,
    read_file,
    write_file,
    cleanup_files,
)
from prompts import (
    ORCHESTRATOR_PROMPT,
    RESEARCHER_PROMPT,
    EDITOR_PROMPT,
)

In [3]:
# -------------------------
# Web Search Tool
# -------------------------

@tool
def web_search(query: str) -> str:
    """
    Perform a live web search using Ollama Cloud Web Search API.

    Input:
        query: search query string

    Output:
        JSON string of top results (max_results=2).
    """
    response = ollama.web_search(query, max_results=2)
    response = response.results

    return response


In [4]:
result = web_search.invoke({"query": "Latest global news"})

In [5]:
result

[WebSearchResult(content='Breaking News, World News and Video from Al Jazeera\nSkip links[Skip to Featured Content](#featured-news-container)[Skip to Content Feed](#news-feed-container)[Skip to Most Read](#most-read-container)\n[\n# Al Jazeera\n, link to home page\n](https://www.aljazeera.com/)\n[\nplay\nLive](https://www.aljazeera.com/live)\nSign up\nShow navigation menu\n[\nplay\nLive](https://www.aljazeera.com/live)\nClick here to searchsearch\nSign up\nFeatured Content\n![Lake Maracaibo](https://www.aljazeera.com/wp-content/uploads/2025/12/GettyImages-2251601503-1766053562.jpg?resize=570%2C380&amp;quality=80)\n### [US confirms seizure of second oil vessel off Venezuela coast](https://www.aljazeera.com/news/2025/12/20/us-seizes-second-oil-vessel-off-venezuela-coast-officials-say)\nThe incident marks the second time in recent weeks that the US has seized an oil tanker near Venezuela.\n[### US, Egypt, Qatar, Turkiye urge restraint in Gaza as Israeli attacks continue\n](https://www.alj

In [6]:
# -------------------------
# LLM
# -------------------------

# llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
llm = ChatGoogleGenerativeAI(model="gemini-3-pro-preview")

In [7]:
# -------------------------
# Worker Agents: Researcher & Editor
# -------------------------
researcher_tools = [ls, write_file, read_file, web_search]

researcher_agent = create_agent(
    model=llm,
    tools=researcher_tools,
    system_prompt=RESEARCHER_PROMPT,
    state_schema=DeepAgentState,
)

editor_tools = [ls, read_file, write_file, cleanup_files]
editor_agent = create_agent(
    model=llm,
    tools=editor_tools,
    system_prompt=EDITOR_PROMPT,
    state_schema=DeepAgentState,
)


In [8]:
# -------------------------
# Orchestrator Tools (call other agents)
# -------------------------
from typing import Annotated
from langgraph.prebuilt import InjectedState
from langchain_core.tools import InjectedToolCallId
from langgraph.types import Command

@tool
def run_researcher(
    theme_id: int,
    thematic_question: str,
    state: Annotated[DeepAgentState, InjectedState],
    max_retries: int = 2
) -> str:
    """
    Run a single Research agent for ONE thematic question.

    Args:
        theme_id: The theme number (1, 2, 3, etc.)
        thematic_question: The specific thematic question to research
        state: Injected agent state providing user_id/thread_id
        max_retries: Number of retry attempts if researcher fails

    The Researcher will:
    - Receive ONE thematic question as input
    - Break it into 2-4 focused search queries
    - Use web_search to gather information
    - Write files to researcher/ folder with hash-based names:
        * researcher/<hash>_theme.md (research findings)
        * researcher/<hash>_sources.txt (raw sources)

    Returns a short status string for the orchestrator.
    """
    from file_tools import generate_hash
    
    # Generate hash for unique file naming
    file_hash = generate_hash(f"{theme_id}_{thematic_question}")
    
    # Build sub-state with theme-specific context
    sub_state: DeepAgentState = {
        "messages": state["messages"] + [
            AIMessage(content=f"[THEME {theme_id}] Research this question: {thematic_question}\n\n"
                             f"File hash: {file_hash}\n"
                             f"Save your findings to: researcher/{file_hash}_theme.md\n"
                             f"Save your sources to: researcher/{file_hash}_sources.txt")
        ],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    
    # Execute researcher with retry logic
    for attempt in range(max_retries + 1):
        try:
            researcher_agent.invoke(sub_state)
            return f"✓ Theme {theme_id} research completed (hash: {file_hash})"
        except Exception as e:
            if attempt < max_retries:
                print(f"⚠ Theme {theme_id} failed (attempt {attempt + 1}/{max_retries + 1}), retrying...")
                continue
            else:
                return f"✗ Theme {theme_id} failed after {max_retries + 1} attempts: {str(e)}"
    
    return f"✓ Theme {theme_id} research completed (hash: {file_hash})"


In [9]:
@tool
def write_research_plan(
    thematic_questions: list[str],
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """
    Write the high-level research plan with major thematic questions.

    Args:
        thematic_questions: List of 3-5 major thematic questions that break down
                           the user's query. These guide the Researcher agent.
        state: Injected agent state providing user_id/thread_id.
        tool_call_id: Tool call ID for attaching a ToolMessage.

    The research_plan.md will be read by the Researcher to guide tactical research.

    Returns:
        Command with a ToolMessage confirming the plan was written.
    """
    from file_tools import _disk_path
    from langchain_core.messages import ToolMessage
    from langgraph.types import Command
    
    # Build content for research_plan.md
    content = "# Research Plan\n\n"
    content += "## User Query\n"
    # Extract user query from messages
    user_msg = [m for m in state["messages"] if hasattr(m, 'content')]
    if user_msg:
        content += f"{user_msg[-1].content}\n\n"
    
    content += "## Thematic Questions\n\n"
    for i, question in enumerate(thematic_questions, 1):
        content += f"{i}. {question}\n"
    
    # Write to disk
    path = _disk_path(state, "research_plan.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    
    msg = f"[RESEARCH PLAN WRITTEN] research_plan.md with {len(thematic_questions)} thematic questions -> {path}"
    return Command(
        update={
            "messages": [ToolMessage(msg, tool_call_id=tool_call_id)]
        }
    )


@tool
def run_editor(
    state: Annotated[DeepAgentState, InjectedState]
) -> str:
    """
    Run the Editor agent for this user/thread.

    The Editor will:
    - read research_plan.md, theme files, sources.txt
    - write the final answer to report.md

    Returns a short status string for the orchestrator.
    """
    sub_state: DeepAgentState = {
        "messages": state["messages"],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    editor_agent.invoke(sub_state)
    return "Editor completed. Final report is written to report.md."

In [10]:
# -------------------------
# Orchestrator Agent
# -------------------------

orchestrator_tools = [
    write_research_plan,  # strategic planning
    run_researcher,       # tactical research
    run_editor,           # synthesis
    cleanup_files,        # resetting workspace
]

orchestrator_agent = create_agent(
    model=llm,
    tools=orchestrator_tools,
    system_prompt=ORCHESTRATOR_PROMPT,
    state_schema=DeepAgentState,
)

In [18]:
# orchestrator_agent
# editor_agent


### Example usage


In [11]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Tell me a joke about computers.")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)
result

{'messages': [HumanMessage(content='Tell me a joke about computers.', additional_kwargs={}, response_metadata={}, id='fa2a350b-ad9a-4787-8bb6-51fb4520e04c'),
  AIMessage(content=[{'type': 'text', 'text': 'Why did the computer go to the doctor?\n\nBecause it had a bad virus!', 'extras': {'signature': 'Ep4DCpsDAXLI2nwYgYCs3tJJS9tM73fW1HbLRfxVPldL8BF5AtnbiC6Oa3hQVXc/pCsk7Qv67ek+srMoxGOFyJcDYa9TpHGpSH8Z/GDNhh2LLh0pUYWa7hwVDNITbDSEfXzVHg6xJ6uMF0DUgyt5TsfFtcCJt7EktirwRAWEn1SxbRpPFViMtuRs2E0qdBby7tei9h+fJgLlJtb0TKR5gCpxnCEaA57z17jk8m3eCtdtl8OUzd7QRPypqkjQF5hPRYvFyd/gBPiOMBDOd1QFvC5HhwzRAm2y4Hr4A9XzrMEHoEsGhJAy1qgZVfkmpub9zQl10L+CZOMAFQ3SO8hAMDzfrZcBIEgEkzr9DpGuYJ1d6Rh62cQnVD9tTs9hwolUsP4EWVu2Uk5X3F+z7q6Gzoan5n9in937YG6PldGhNKREo+sIsAEug7DCWSVz8d2GyKSbPgP7C6+qCkhqO3GMYLyP6WMRuFfPJvFFzSyFuO6mV8UUPMhnFvFZQf/xdLZD3YOfdK7lN3S1SuV6FmWTxAd/Pa1PV9rpS7y90rkoi7Kt'}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_n

In [12]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Do a detailed, well-structured analysis of MCP including history")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)

⚠ Theme 5 failed (attempt 1/4), retrying...
⚠ Theme 5 failed (attempt 2/4), retrying...
⚠ Theme 5 failed (attempt 3/4), retrying...
⚠ Theme 1 failed (attempt 1/4), retrying...
⚠ Theme 1 failed (attempt 2/4), retrying...
⚠ Theme 1 failed (attempt 3/4), retrying...


In [13]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured analysis of MCP including history', additional_kwargs={}, response_metadata={}, id='7a11b128-aa4f-480e-98f2-acd0d9770b2b'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'run_researcher', 'arguments': '{"thematic_question": "What are the advantages, limitations, and future outlook for MCP adoption?", "max_retries": 3, "theme_id": 5}'}, '__gemini_function_call_thought_signatures__': {'c2a7d19c-f990-4326-a794-80d7e623f667': 'ErcQCrQQAXLI2nwnddvg8TUFoFOqmrFDcz8wlteiGKPGSbC820e+ARKNSmCD9uyNevoUK0O9miyWyfy1UAvojGI1LQ7QRoQcUu1x3EZ4Duotmu6h137DesO1e6siBncrwSqReD9WHilaMSWRYV5TSvLMmSiRgraC+Qdi9Tw/7vngh/afHDw7J+r3fSaP8PlJenluiauJYqsB8Q5HObfDI8vKPiwBj2EkkBi2lZc3yBg+O+7FmMKoS4mSb4dKBwM4bZo9lcllXdT7ocxTHihQ4uoOae0D2u0pFWaxmSakLzh2VKmKnaAGdaI3pTeBJrbxYFOJD4Ci2hB7K2cSFajW8WmB1QbVxZL1hpwdtOMyOT+N+6bbYCW520G6yR8mjsMm51dDAtDpM7eq8wc0akrDhovEtvZDXaGhuEpw+7E09dU9EPTt6sJXSDxAwHsZF8858QrmlC/ttxKZG6+GScjmvg+KpkIzCQvk0NQ

In [14]:
# from langchain.messages import HumanMessage

# state = {
#         "messages": [HumanMessage(content="Do a detailed, well-structured comparison of MCP and API.")],
#         "user_id": "user_123",
#         "thread_id": "thread_mcp_002",
#     }

# result = orchestrator_agent.invoke(state)

In [15]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured analysis of MCP including history', additional_kwargs={}, response_metadata={}, id='7a11b128-aa4f-480e-98f2-acd0d9770b2b'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'run_researcher', 'arguments': '{"thematic_question": "What are the advantages, limitations, and future outlook for MCP adoption?", "max_retries": 3, "theme_id": 5}'}, '__gemini_function_call_thought_signatures__': {'c2a7d19c-f990-4326-a794-80d7e623f667': 'ErcQCrQQAXLI2nwnddvg8TUFoFOqmrFDcz8wlteiGKPGSbC820e+ARKNSmCD9uyNevoUK0O9miyWyfy1UAvojGI1LQ7QRoQcUu1x3EZ4Duotmu6h137DesO1e6siBncrwSqReD9WHilaMSWRYV5TSvLMmSiRgraC+Qdi9Tw/7vngh/afHDw7J+r3fSaP8PlJenluiauJYqsB8Q5HObfDI8vKPiwBj2EkkBi2lZc3yBg+O+7FmMKoS4mSb4dKBwM4bZo9lcllXdT7ocxTHihQ4uoOae0D2u0pFWaxmSakLzh2VKmKnaAGdaI3pTeBJrbxYFOJD4Ci2hB7K2cSFajW8WmB1QbVxZL1hpwdtOMyOT+N+6bbYCW520G6yR8mjsMm51dDAtDpM7eq8wc0akrDhovEtvZDXaGhuEpw+7E09dU9EPTt6sJXSDxAwHsZF8858QrmlC/ttxKZG6+GScjmvg+KpkIzCQvk0NQ